In [ ]:
!pip install sentence-transformers umap-learn
from google.colab import files
uploaded_files = files.upload() # upload the local files from the computer

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import sys
from sentence_transformers import SentenceTransformer # the SBERT model
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
sys.path.insert(0,'/content') # look in /content for any local files we uploaded 

In [ ]:
df = pd.read_csv('/content/data_preprocessed_csv.csv')

df['cleaned_abstract'] = df['cleaned_abstract'].fillna('') # handling missing values
df['abstract'] = df['abstract'].fillna('')

print(f"Total number of papers : {len(df)}")
print(f"Number of columns : {df.columns.tolist()}")

In [ ]:
model_used = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Extracts abstracts and encode them into numerical embeddings using the model
abstracts = df['abstract'].tolist() 
embeddings = model_used.encode(abstracts,batch_size=64,convert_to_numpy=True,device='cuda')
print(f"Embeddings shape : {embeddings.shape}")

In [ ]:
np.save('/content/embeddings.npy',embeddings) # save the embeddings to the computer
files.download('/content/embeddings.npy')

In [ ]:
similarity_score = cosine_similarity(embeddings[0:1],embeddings[1:2])[0][0] # Picking 2 random papers and checking their similarity score
print("\nPaper 1 is about :",df['abstract'].iloc[0][:400]) 
print("\nPaper 2 is about :",df['abstract'].iloc[1][:400])
print(f"Their similarity score is : {similarity_score : }") # Range betwen 0 to 1, higher means more similar

In [ ]:
def table_of_similar_papers(query, top_k=10):
    query_embedding = model_used.encode([query],convert_to_numpy=True,device='cuda') #turning our search query into 384 numbers
    scores = cosine_similarity(query_embedding,embeddings)[0] # comparing query against every other paper's embedding
    top_results = scores.argsort()[::-1][:top_k] # grabbing top results after sorting
    results = df.iloc[top_results][['title','year','abstract']].copy()
    results['similarity score'] = scores[top_results].round(4)
    results['abstract'] = results['abstract'].str[:300]
    return results.reset_index(drop=True)

In [ ]:
table_of_similar_papers("deep learning") 

In [ ]:
table_of_similar_papers("natural language processing")